# Análisis de artículos de Wikipedia, por cliente

Este es el notebook del enunciado adaptado para verificar el sistema distribuido. Lee los archivos `.parquet` de un cliente y exporta los resultados de Q1 a Q5 a CSV, para compararlos con las salidas del sistema.

**Cómo usarlo:** cambiá `CLIENT` en la primera celda (0, 1 o 2) y ejecutá todo con *Kernel → Restart Kernel and Run All Cells*. Los archivos quedan en `output_notebook/client_N/`.

**Diferencias con el original** (para que los resultados coincidan con los del sistema, que está escrito en Go):
- Lee los archivos del cliente en vez de bajarlos de Kaggle.
- Reemplaza `İ` por `i` y `Σ` por `σ` antes de pasar a minúsculas: el `lower()` de Python parte la `İ` turca en dos letras y usa `ς` para la sigma al final de palabra, y el `ToLower` de Go no.
- El top 20 de Q4 desempata por palabra ascendente (el original desempata por orden de aparición).
- Carga los archivos de forma más liviana (mismos resultados): concatena una sola vez y descarta `sections`, `image` y `name` de los artículos posteriores a 2020, que no se usan.
- Al final exporta los resultados a CSV (`q1.csv` a `q5.csv`, `q5_average.txt`).

In [ ]:
import os

CLIENT = 0  # cambiar a 1 o 2 y volver a ejecutar todo
CLIENT = int(os.environ.get("CLIENT", CLIENT))
DATASET_DIR = os.environ.get("DATASET_DIR", f"/data/client_{CLIENT}")
EXPORT_DIR = os.environ.get("EXPORT_DIR", f"/repo/output_notebook/client_{CLIENT}")
print(f"Cliente {CLIENT}: leo {DATASET_DIR} y exporto en {EXPORT_DIR}")

In [ ]:
import re
import os
import json
import numpy
import collections

import pandas as pd
from nltk.corpus import stopwords

In [ ]:
# Las stopwords de NLTK vienen incluidas en la imagen Docker (scripts/Dockerfile)
import nltk

In [ ]:
import glob


COLUMNS_TO_DROP = ['additional_entities', 'date_created',
       'description', 'event', 'identifier',
       'infoboxes', 'is_part_of', 'license', 'main_entity',
       'references', 'tables', 'version']

import pyarrow.parquet as pq

frames = []
for path in sorted(glob.glob(os.path.join(DATASET_DIR, "**", "*.parquet"), recursive=True)):
    columns = [name for name in pq.read_schema(path).names if name not in COLUMNS_TO_DROP]
    df = pd.read_parquet(path, columns=columns)
    recent = df["date_modified"].dt.year > 2020
    for heavy in ("sections", "image", "name"):
        if heavy in df.columns:
            df[heavy] = df[heavy].where(~recent, None)
    frames.append(df)
wiki_entry_df = pd.concat(frames)
del frames

wiki_entry_df.info()

In [ ]:
wiki_entry_df.head()

In [ ]:
# Dataset cleanup
def in_language_parse(in_language_column):
    if not isinstance(in_language_column, dict):
        return None
    return in_language_column.get('identifier', None)
wiki_entry_df["in_language"] = wiki_entry_df["in_language"].apply(in_language_parse)

def image_object_to_url(image):
    if not isinstance(image, dict):
        return None
    return image.get('content_url', None)
wiki_entry_df["image"] = wiki_entry_df["image"].apply(image_object_to_url)

In [ ]:
# Entries modified before or at 2020
wiki_entry_df['date_modified_year'] = wiki_entry_df['date_modified'].dt.year
wiki_entry_before_2020_df = wiki_entry_df[wiki_entry_df['date_modified_year'] <= 2020].copy()

In [ ]:
# Url and date of modification of english entries before 2020
wiki_entry_before_2020_df[wiki_entry_before_2020_df["in_language"] == "en"][["date_modified", "url"]]

In [ ]:
# Name, date of modification and image of all articles that were modified before 2020 and have a biography/biographie section
BIOGRAPHY_SECTION_NAMES = ['biographie', 'biography']
def has_biography_section(sections):
    if not isinstance(sections, str):
        return False
    section_names =  [section.get("name", "").lower() for section in json.loads(sections)]
    for biography_section_name in BIOGRAPHY_SECTION_NAMES:
        if biography_section_name in section_names:
            return True
    return False
wiki_entry_before_2020_df['has_biography_section'] = wiki_entry_before_2020_df['sections'].apply(has_biography_section)

#Note: Actual implementation should download the image from the url
wiki_entry_before_2020_df[(wiki_entry_before_2020_df['has_biography_section'] == True)][["name", "date_modified", "image"]]

In [ ]:
# Word and count of the 20 most common words in the abstract
TOP_SIZE = 20
WORD_REGEX = r'\b\w+\b'
NUMBER_REGEX = r'[+-]?((\d+\.?\d*)|(\.\d+))'
# Source and discussion: https://stackoverflow.com/questions/12643009/regular-expression-for-floating-point-numbers/42629198#42629198

def extract_non_stopwords_from_abstract(abstract):
    if not isinstance(abstract, str):
        return []
    words = re.findall(WORD_REGEX, abstract.replace('İ', 'i').replace('Σ', 'σ').lower())
    return [word for word in words if word not in stop_words and not re.fullmatch(NUMBER_REGEX, word)]

stop_words = set(stopwords.words('english')) | set(stopwords.words('french'))
words_counter = collections.Counter()
for abstract in wiki_entry_df['abstract'].dropna():
    non_stop_words = extract_non_stopwords_from_abstract(abstract)
    words_counter.update(set(non_stop_words))
top_words = sorted(words_counter.items(), key=lambda item: (-item[1], item[0]))[:TOP_SIZE]
pd.DataFrame(top_words, columns=["word", "count"])

In [ ]:
# Shared words between all french and english articles in the abstract
def fill_language_word_set(df, language):
    language_words = set()
    for abstract in df[df["in_language"] == language]['abstract'].dropna():
        language_words |= set(extract_non_stopwords_from_abstract(abstract))
    return language_words

english_words = fill_language_word_set(wiki_entry_df, "en")
french_words = fill_language_word_set(wiki_entry_df, "fr")
shared_words = english_words & french_words
pd.Series(data=list(english_words & french_words))

In [ ]:
# Entries after 2020
wiki_entry_after_2020_df = wiki_entry_df[wiki_entry_df['date_modified_year'] > 2020].copy()

In [ ]:
# Url and count of all articles modified after 2020 with an abstract word count greater than the average of all articles written before or at 2020
def calc_abstract_word_count_average(df):
    count = 0
    total = 0
    for abstract in df['abstract'].dropna():
        total += len(extract_non_stopwords_from_abstract(abstract))
        count += 1
    return total/count

before_2020_abstract_average = calc_abstract_word_count_average(wiki_entry_before_2020_df)
print("Before 2020 abstract average:", before_2020_abstract_average)

In [ ]:
wiki_entry_after_2020_df["abstract_word_count"] = wiki_entry_after_2020_df["abstract"].apply(lambda abstract: len(extract_non_stopwords_from_abstract(abstract)))
wiki_entry_after_2020_df[wiki_entry_after_2020_df["abstract_word_count"] > before_2020_abstract_average][["url","abstract_word_count"]]

In [ ]:
os.makedirs(EXPORT_DIR, exist_ok=True)

q1 = wiki_entry_before_2020_df[wiki_entry_before_2020_df["in_language"] == "en"]
seconds = (pd.to_datetime(q1["date_modified"], utc=True) - pd.Timestamp("1970-01-01", tz="UTC")) // pd.Timedelta(seconds=1)
pd.DataFrame({"url": q1["url"], "date_modified": seconds}).to_csv(f"{EXPORT_DIR}/q1.csv", index=False)

pd.DataFrame(top_words, columns=["word", "count"]).to_csv(f"{EXPORT_DIR}/q4.csv", index=False)

pd.Series(sorted(shared_words)).to_csv(f"{EXPORT_DIR}/q3.csv", index=False, header=False)

q2 = wiki_entry_before_2020_df[wiki_entry_before_2020_df["has_biography_section"] & wiki_entry_before_2020_df["image"].notna()]
q2[["url", "name"]].to_csv(f"{EXPORT_DIR}/q2.csv", index=False)

try:
    q5 = wiki_entry_after_2020_df[wiki_entry_after_2020_df["abstract_word_count"] > before_2020_abstract_average][["url", "abstract_word_count"]]
    q5.to_csv(f"{EXPORT_DIR}/q5.csv", index=False)
    open(f"{EXPORT_DIR}/q5_average.txt", "w").write(str(before_2020_abstract_average))
except NameError as error:
    open(f"{EXPORT_DIR}/q5_error.txt", "w").write(f"El notebook no pudo calcular Q5: {error}")

print("Archivos exportados en", EXPORT_DIR, ":", sorted(os.listdir(EXPORT_DIR)))